# FedSafe-Fuse — Multi-seed sweep (F2)
Adds 5-seed **mean±std** to the headline MedMNIST + BraTS fusion tables.

**Run cells top-to-bottom.** The sweep is *resumable*: after each seed the CSV is saved to local disk **and** Drive, and completed seeds are skipped on re-run — so a Colab disconnect loses at most one in-progress seed. Total runtime ~2.5–3.5 h; split across sessions freely.

Model / loss / trainer code is copied **verbatim** from `03_fedtrain` and `10b_brats`; seed 42 runs first and is self-checked against the published single-seed numbers.

In [ ]:
# Multi-seed sweep (F2): 5-seed mean+/-std for the headline fusion tables
import os, sys, time, json, random, copy, csv, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEEDS = [42, 1, 2, 3, 4]   # 42 first == self-check vs published single-seed numbers
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| Torch:', torch.__version__, '| Seeds:', SEEDS)

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


In [ ]:
# Mount Drive + mirror MedMNIST npz, BraTS case files, and partitions to local disk
from google.colab import drive
import shutil
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/FedSafeFuse'
LOCAL_ROOT = '/content/local_fedsafe'
for sub in ['medmnist', 'partitions', 'brats/processed', 'checkpoints', 'results', 'figures']:
    os.makedirs(f'{LOCAL_ROOT}/{sub}', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)

# Partitions
for fname in ['medmnist_K3.json', 'calibration_manifest.json', 'brats_K3.json']:
    s = f'{DRIVE_ROOT}/partitions/{fname}'; d = f'{LOCAL_ROOT}/partitions/{fname}'
    if os.path.exists(s) and not os.path.exists(d):
        shutil.copy(s, d)

# MedMNIST paired arrays (hundreds of MB; first run only)
for fname in ['train_paired.npz', 'test_paired.npz']:
    s = f'{DRIVE_ROOT}/medmnist/{fname}'; d = f'{LOCAL_ROOT}/medmnist/{fname}'
    if not os.path.exists(d):
        t0 = time.time(); print(f'Copying {fname} ...'); shutil.copy(s, d)
        print(f'  {fname}: {os.path.getsize(d)/1e6:.0f} MB in {time.time()-t0:.1f}s')

# BraTS: mirror only the case files the partition references (12/client + 12 calib)
with open(f'{LOCAL_ROOT}/partitions/brats_K3.json') as f:
    _bpart = json.load(f)
CASES_PER_CLIENT = 12; CALIB_CASES = 12
brats_client_case_lists = [cl[:CASES_PER_CLIENT] for cl in _bpart['client_cases']]
brats_calib_case_list = _bpart['calibration_cases'][:CALIB_CASES]
_needed = sorted(set([c for cl in brats_client_case_lists for c in cl] + brats_calib_case_list))
t0 = time.time(); copied = 0
for cid in _needed:
    s = f'{DRIVE_ROOT}/brats/processed/{cid}.npz'; d = f'{LOCAL_ROOT}/brats/processed/{cid}.npz'
    if os.path.exists(s) and not os.path.exists(d):
        shutil.copy(s, d); copied += 1
print(f'Mirrored {copied} BraTS case files ({len(_needed)} needed) in {time.time()-t0:.1f}s')
PROJECT_DRIVE = LOCAL_ROOT
print('PROJECT_DRIVE:', PROJECT_DRIVE)


## Shared building blocks (verbatim from source notebooks)

In [ ]:
# FedSafe-Fuse model
"""FedSafe-Fuse: dual MobileNetV3-Small encoders + 2-layer Transformer + conv decoder.

Per the project proposal:
    - Two modality-specific encoders, each MobileNetV3-Small (~2.5M params total each)
    - Shared 2-layer Transformer module (2 heads, embed dim 128) for cross-modal attention
    - Convolutional decoder
    - Target ~8M params

Actual realised parameter count depends on decoder width; default config lands at ~5M.
"""

from __future__ import annotations

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small


class ModalityEncoder(nn.Module):
    """MobileNetV3-Small features adapted to 1-channel input.

    MobileNetV3-Small downsamples by 32x, so a 128x128 input yields 4x4 features.
    We bilinearly upsample to 8x8 to give the cross-modal Transformer 64 tokens
    per modality (matching the proposal). The upsample is parameter-free.
    """

    def __init__(self, embed_dim: int = 128):
        super().__init__()
        base = mobilenet_v3_small(weights=None)
        # Replace first conv to accept a single channel (medical grayscale)
        base.features[0][0] = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1, bias=False)
        self.features = base.features  # (B, 576, 4, 4) for 128x128 input
        self.proj = nn.Conv2d(576, embed_dim, kernel_size=1)
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.features(x)
        h = self.proj(h)
        return self.upsample(h)  # (B, embed_dim, 8, 8)


class CrossModalTransformer(nn.Module):
    """2-layer Transformer encoder over concatenated MRI/PET tokens.

    Input feature maps are flattened to 64 spatial tokens per modality, given a
    learned modality embedding, concatenated to 128 tokens, given a learned
    positional embedding, and passed through `num_layers` transformer encoder layers.
    Output is averaged across the two modality halves and reshaped back to a
    spatial feature map for the decoder.
    """

    def __init__(
        self,
        embed_dim: int = 128,
        num_layers: int = 2,
        num_heads: int = 2,
        ff_dim: int = 256,
        dropout: float = 0.1,
        spatial_tokens_per_modality: int = 64,
    ):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        total_tokens = spatial_tokens_per_modality * 2
        self.pos_embed = nn.Parameter(torch.zeros(1, total_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.mod_embed = nn.Embedding(2, embed_dim)  # 0 = MRI, 1 = PET
        self.spatial_tokens = spatial_tokens_per_modality

    def forward(self, mri_feat: torch.Tensor, pet_feat: torch.Tensor) -> torch.Tensor:
        # mri_feat, pet_feat: (B, C, H, W) with H*W = self.spatial_tokens
        B, C, H, W = mri_feat.shape
        mri_tokens = mri_feat.flatten(2).transpose(1, 2)  # (B, HW, C)
        pet_tokens = pet_feat.flatten(2).transpose(1, 2)
        mri_tokens = mri_tokens + self.mod_embed.weight[0]
        pet_tokens = pet_tokens + self.mod_embed.weight[1]
        tokens = torch.cat([mri_tokens, pet_tokens], dim=1) + self.pos_embed
        out = self.transformer(tokens)
        mri_out = out[:, : self.spatial_tokens].transpose(1, 2).reshape(B, C, H, W)
        pet_out = out[:, self.spatial_tokens :].transpose(1, 2).reshape(B, C, H, W)
        return 0.5 * (mri_out + pet_out)


class FusionDecoder(nn.Module):
    """Progressive bilinear upsample + conv decoder: 8x8 -> 16 -> 32 -> 64 -> 128."""

    def __init__(self, embed_dim: int = 128, base_ch: int = 256):
        super().__init__()
        c0, c1, c2, c3 = base_ch, base_ch, base_ch // 2, base_ch // 4
        self.up1 = self._block(embed_dim, c0)
        self.up2 = self._block(c0, c1)
        self.up3 = self._block(c1, c2)
        self.up4 = self._block(c2, c3)
        self.final = nn.Conv2d(c3, 1, kernel_size=1)

    @staticmethod
    def _block(in_ch: int, out_ch: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.up1(x)
        h = self.up2(h)
        h = self.up3(h)
        h = self.up4(h)
        return torch.sigmoid(self.final(h))


class FedSafeFuse(nn.Module):
    """End-to-end FedSafe-Fuse model. Inputs: two (B, 1, 128, 128) tensors. Output: (B, 1, 128, 128)."""

    def __init__(
        self,
        embed_dim: int = 128,
        transformer_layers: int = 2,
        transformer_heads: int = 2,
        transformer_ff: int = 256,
        decoder_base_ch: int = 256,
    ):
        super().__init__()
        self.mri_encoder = ModalityEncoder(embed_dim)
        self.pet_encoder = ModalityEncoder(embed_dim)
        self.fusion = CrossModalTransformer(
            embed_dim=embed_dim,
            num_layers=transformer_layers,
            num_heads=transformer_heads,
            ff_dim=transformer_ff,
        )
        self.decoder = FusionDecoder(embed_dim=embed_dim, base_ch=decoder_base_ch)

    def forward(self, mri: torch.Tensor, pet: torch.Tensor) -> torch.Tensor:
        mri_feat = self.mri_encoder(mri)
        pet_feat = self.pet_encoder(pet)
        fused_feat = self.fusion(mri_feat, pet_feat)
        return self.decoder(fused_feat)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



In [ ]:
# IFCNN baseline
"""IFCNN baseline (Zhang et al. 2021).

A general image fusion framework based on a two-stream CNN with element-wise fusion
followed by reconstruction. Simplified from the original paper for our 128x128
single-channel fusion setting. Used as Baseline 2 in the FedSafe-Fuse evaluation.
"""

from __future__ import annotations

import torch
import torch.nn as nn


class IFCNN(nn.Module):
    """Two-stream conv + element-wise fusion + conv reconstruction."""

    FUSION_MODES = ("max", "mean", "sum")

    def __init__(self, base_ch: int = 64, fusion: str = "max"):
        super().__init__()
        assert fusion in self.FUSION_MODES, f"fusion must be one of {self.FUSION_MODES}"
        self.fusion_mode = fusion
        self.stream1 = self._make_stream(base_ch)
        self.stream2 = self._make_stream(base_ch)
        self.recon = nn.Sequential(
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, base_ch // 2, 3, padding=1),
            nn.BatchNorm2d(base_ch // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch // 2, 1, kernel_size=1),
        )

    @staticmethod
    def _make_stream(c: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Conv2d(1, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        f1 = self.stream1(x1)
        f2 = self.stream2(x2)
        if self.fusion_mode == "max":
            fused = torch.maximum(f1, f2)
        elif self.fusion_mode == "mean":
            fused = 0.5 * (f1 + f2)
        else:
            fused = f1 + f2
        return torch.sigmoid(self.recon(fused))


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



In [ ]:
# Composite fusion loss + eval metrics
"""Composite fusion loss: L1 + beta * (1 - SSIM).

Implements a differentiable SSIM via a Gaussian-windowed convolution
(no external dependency on pytorch_msssim). Designed for single-channel
medical images in [0, 1].
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


def _gaussian_window(window_size: int, sigma: float, channels: int) -> torch.Tensor:
    coords = torch.arange(window_size, dtype=torch.float32) - (window_size - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w_2d = g.unsqueeze(0) * g.unsqueeze(1)  # (W, W)
    return w_2d.expand(channels, 1, window_size, window_size).contiguous()


class SSIMLoss(nn.Module):
    """1 - SSIM, computed with a Gaussian sliding window. Single-channel input in [0, 1]."""

    def __init__(
        self,
        window_size: int = 11,
        sigma: float = 1.5,
        data_range: float = 1.0,
        channels: int = 1,
    ):
        super().__init__()
        self.window_size = window_size
        self.data_range = data_range
        self.C1 = (0.01 * data_range) ** 2
        self.C2 = (0.03 * data_range) ** 2
        self.register_buffer("window", _gaussian_window(window_size, sigma, channels))

    def _filter(self, x: torch.Tensor) -> torch.Tensor:
        return F.conv2d(x, self.window, padding=self.window_size // 2, groups=x.shape[1])

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        mu_p = self._filter(pred)
        mu_t = self._filter(target)
        mu_p2 = mu_p ** 2
        mu_t2 = mu_t ** 2
        mu_pt = mu_p * mu_t
        sigma_p2 = self._filter(pred * pred) - mu_p2
        sigma_t2 = self._filter(target * target) - mu_t2
        sigma_pt = self._filter(pred * target) - mu_pt
        num = (2 * mu_pt + self.C1) * (2 * sigma_pt + self.C2)
        den = (mu_p2 + mu_t2 + self.C1) * (sigma_p2 + sigma_t2 + self.C2)
        ssim_map = num / den
        return 1.0 - ssim_map.mean()


class FusionLoss(nn.Module):
    """L1(fused, ref) + beta * SSIMLoss(fused, ref).

    Reference composite is a per-pixel weighted average of source modalities, per the proposal.
    Default weights are (0.5, 0.5). Override `reference_weights` per dataset if needed.
    """

    def __init__(self, beta: float = 1.0, reference_weights: tuple[float, float] = (0.5, 0.5)):
        super().__init__()
        assert abs(sum(reference_weights) - 1.0) < 1e-6, "reference_weights must sum to 1.0"
        self.l1 = nn.L1Loss()
        self.ssim = SSIMLoss()
        self.beta = beta
        w1, w2 = reference_weights
        self.register_buffer("w1", torch.tensor(w1))
        self.register_buffer("w2", torch.tensor(w2))

    def reference(self, src1: torch.Tensor, src2: torch.Tensor) -> torch.Tensor:
        return self.w1 * src1 + self.w2 * src2

    def forward(
        self,
        fused: torch.Tensor,
        src1: torch.Tensor,
        src2: torch.Tensor,
    ) -> torch.Tensor:
        ref = self.reference(src1, src2)
        return self.l1(fused, ref) + self.beta * self.ssim(fused, ref)


@torch.no_grad()
def ssim_score(pred: torch.Tensor, target: torch.Tensor) -> float:
    """SSIM (higher is better) for evaluation. Mirrors SSIMLoss without the 1 - ... wrap."""
    loss_fn = SSIMLoss().to(pred.device)
    return float(1.0 - loss_fn(pred, target).item())


@torch.no_grad()
def psnr_score(pred: torch.Tensor, target: torch.Tensor, data_range: float = 1.0) -> float:
    """PSNR in dB."""
    mse = F.mse_loss(pred, target).item()
    if mse == 0:
        return float("inf")
    return 10.0 * torch.log10(torch.tensor(data_range ** 2 / mse)).item()


## Part A — MedMNIST sweep

In [ ]:
# MedMNIST: load partition + paired arrays ONCE; build shared-array datasets (seed-independent)
class _SharedArrayMedMNIST(torch.utils.data.Dataset):
    def __init__(self, mri, pet, labels, indices):
        self.mri = mri; self.pet = pet; self.labels = labels; self.indices = list(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        gi = self.indices[i]
        return (torch.from_numpy(self.mri[gi]).unsqueeze(0),
                torch.from_numpy(self.pet[gi]).unsqueeze(0), int(self.labels[gi]))

with open(f'{PROJECT_DRIVE}/partitions/medmnist_K3.json') as f:
    medmnist_part = json.load(f)
with np.load(f'{PROJECT_DRIVE}/medmnist/train_paired.npz') as d:
    train_mri = d['mri']; train_pet = d['pet']; train_labels = d['labels'].astype(np.int64).flatten()
with np.load(f'{PROJECT_DRIVE}/medmnist/test_paired.npz') as d:
    test_mri = d['mri']; test_pet = d['pet']; test_labels = d['labels'].astype(np.int64).flatten()
medmnist_client_datasets = [_SharedArrayMedMNIST(train_mri, train_pet, train_labels, ci)
                            for ci in medmnist_part['client_indices']]
medmnist_eval_dataset = _SharedArrayMedMNIST(test_mri, test_pet, test_labels, list(range(128)))
print('MedMNIST client sizes:', [len(ds) for ds in medmnist_client_datasets],
      '| eval:', len(medmnist_eval_dataset))


In [ ]:
# FedAvg simulator (standard / dpsgd / fipca modes)
"""In-process FedAvg simulator supporting standard, DP-SGD, and FIPCA modes.

A single-process replacement for Flower's SimulationEngine, designed for Colab
Free's compute budget. Mirrors the round structure of Flower so the algorithm
can be swapped to Flower without changing the experimental protocol.

Three modes:
    - 'standard': vanilla FedAvg (raw weight deltas).
    - 'dpsgd'   : DP-SGD on each client (per-batch gradient clip + Gaussian noise),
                  then standard FedAvg aggregation of resulting weight deltas.
    - 'fipca'   : Client weight delta is projected to a rank-d subspace via a
                  pseudo-random Gaussian matrix derived from a fixed seed (shared
                  basis across clients, never transmitted). Server averages the
                  rank-d coefficients and reconstructs.

Communication cost per round is logged in bytes, which is the primary privacy /
efficiency claim of FedSafe-Fuse.
"""

from __future__ import annotations

import time
from typing import Callable, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset


def _flatten_state(state: dict) -> torch.Tensor:
    """Concatenate all tensors in a state_dict into a single 1-D float32 vector."""
    return torch.cat([v.flatten().float() for v in state.values()])


def _unflatten_to_state(flat: torch.Tensor, template: dict) -> dict:
    """Inverse of _flatten_state, using template state_dict for shapes."""
    out: dict = {}
    idx = 0
    for k, v in template.items():
        n = v.numel()
        out[k] = flat[idx : idx + n].reshape(v.shape).to(v.dtype)
        idx += n
    return out


class FedAvgTrainer:
    """Single-process federated trainer.

    Parameters
    ----------
    build_model_fn :
        Zero-arg callable that returns a fresh model on CPU. Cloned per client.
    loss_fn_factory :
        Zero-arg callable returning a loss module (e.g. FusionLoss(beta)).
        Recreated per client to avoid sharing buffers (SSIM Gaussian window).
    client_datasets :
        List of K Dataset objects, one per client. Items must be (mri, pet, ...).
    eval_dataset :
        Held-out evaluation set (e.g. the calibration split).
    device :
        'cuda' or 'cpu'.
    mode :
        'standard', 'dpsgd', or 'fipca'.
    samples_per_local_epoch :
        Cap on examples drawn per client per local epoch (Round 1 compute cut).
    fipca_rank :
        d, the rank of the FIPCA random-projection subspace.
    dp_clip, dp_sigma :
        DP-SGD per-batch gradient L2-clip threshold and noise multiplier.
    lr :
        Adam learning rate.
    batch_size :
        Per-client batch size.
    seed :
        Random seed for reproducibility.
    """

    VALID_MODES = ("standard", "dpsgd", "fipca")

    def __init__(
        self,
        build_model_fn: Callable[[], nn.Module],
        loss_fn_factory: Callable[[], nn.Module],
        client_datasets: list,
        eval_dataset: Dataset,
        device: str = "cuda",
        mode: str = "standard",
        samples_per_local_epoch: int = 100,
        fipca_rank: int = 32,
        dp_clip: float = 1.0,
        dp_sigma: float = 0.5,
        lr: float = 1e-3,
        batch_size: int = 16,
        seed: int = 42,
    ):
        assert mode in self.VALID_MODES, f"mode must be one of {self.VALID_MODES}"
        self.build_model_fn = build_model_fn
        self.loss_fn_factory = loss_fn_factory
        self.client_datasets = client_datasets
        self.eval_dataset = eval_dataset
        self.device = device
        self.mode = mode
        self.samples_per_local_epoch = samples_per_local_epoch
        self.fipca_rank = fipca_rank
        self.dp_clip = dp_clip
        self.dp_sigma = dp_sigma
        self.lr = lr
        self.batch_size = batch_size
        self.seed = seed
        self._rng = np.random.default_rng(seed)

        # Global model (server's authoritative copy)
        self.global_model = build_model_fn().to(device)
        self.global_state = {k: v.detach().clone() for k, v in self.global_model.state_dict().items()}

        # FIPCA: deterministic ORTHONORMAL projection matrix P in (rank, n_params).
        # Built via QR decomposition so that P @ P.T = I_rank exactly, which makes
        # P.T @ P a true rank-d orthogonal projector with operator norm 1.
        # Earlier scaled-Gaussian attempt had operator norm ~sqrt(n_params/rank),
        # causing reconstructed deltas to blow up and Adam to diverge (NaN).
        if mode == "fipca":
            n_params = sum(v.numel() for v in self.global_state.values())
            gen = torch.Generator(device=device).manual_seed(seed)
            G = torch.randn(n_params, fipca_rank, device=device, generator=gen)
            Q, _ = torch.linalg.qr(G)  # Q has orthonormal columns: (n_params, rank)
            self.P = Q.T.contiguous()  # (rank, n_params) with orthonormal rows

    # ------------------------------------------------------------------
    # Client-side local training
    # ------------------------------------------------------------------
    def _local_subset_loader(self, client_idx: int) -> DataLoader:
        ds = self.client_datasets[client_idx]
        n = min(self.samples_per_local_epoch, len(ds))
        sel = self._rng.choice(len(ds), size=n, replace=False)
        subset = Subset(ds, sel.tolist())
        return DataLoader(subset, batch_size=self.batch_size, shuffle=True, num_workers=0)

    def _train_client(self, client_idx: int, n_local_epochs: int) -> dict:
        """Run E local epochs on a fresh random subset of this client's data, return weight delta."""
        local_model = self.build_model_fn().to(self.device)
        local_model.load_state_dict(self.global_state)
        opt = torch.optim.Adam(local_model.parameters(), lr=self.lr, weight_decay=1e-4)
        loss_fn = self.loss_fn_factory().to(self.device)

        local_model.train()
        for _ in range(n_local_epochs):
            loader = self._local_subset_loader(client_idx)
            for batch in loader:
                mri, pet = batch[0].to(self.device), batch[1].to(self.device)
                fused = local_model(mri, pet)
                loss = loss_fn(fused, mri, pet)
                opt.zero_grad()
                loss.backward()
                if self.mode == "dpsgd":
                    torch.nn.utils.clip_grad_norm_(local_model.parameters(), self.dp_clip)
                    for p in local_model.parameters():
                        if p.grad is not None:
                            p.grad.add_(torch.randn_like(p.grad) * self.dp_sigma * self.dp_clip)
                opt.step()

        local_state = local_model.state_dict()
        delta = {k: (local_state[k].detach() - self.global_state[k]).to(self.device) for k in self.global_state}
        return delta

    # ------------------------------------------------------------------
    # Server-side aggregation
    # ------------------------------------------------------------------
    def _compress_delta(self, delta: dict) -> tuple:
        """Return (payload, bytes_sent) for one client's update."""
        if self.mode == "fipca":
            flat = _flatten_state(delta).to(self.device)
            coeffs = self.P @ flat  # (rank,)
            payload = coeffs.detach().clone()
            bytes_sent = payload.numel() * 4  # float32
            return payload, bytes_sent
        else:
            payload = {k: v.detach().clone() for k, v in delta.items()}
            bytes_sent = sum(v.numel() * 4 for v in payload.values())
            return payload, bytes_sent

    def _aggregate(self, client_payloads: list) -> dict:
        if self.mode == "fipca":
            avg_coeffs = torch.stack(client_payloads).mean(dim=0)  # (rank,)
            recon_flat = self.P.T @ avg_coeffs  # (n_params,)
            return _unflatten_to_state(recon_flat, self.global_state)
        else:
            avg = {}
            for k in self.global_state:
                stacked = torch.stack([cp[k] for cp in client_payloads])
                if stacked.dtype.is_floating_point or stacked.dtype.is_complex:
                    avg[k] = stacked.mean(dim=0)
                else:
                    # Integer buffers (e.g. BatchNorm.num_batches_tracked) cannot be
                    # mean()-ed directly; promote to float, mean, and cast back.
                    avg[k] = stacked.to(torch.float32).mean(dim=0).to(stacked.dtype)
            return avg

    # ------------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------------
    @torch.no_grad()
    def evaluate(self, ssim_score_fn, psnr_score_fn) -> dict:
        self.global_model.load_state_dict(self.global_state)
        self.global_model.eval()
        loss_fn = self.loss_fn_factory().to(self.device)
        losses, ssims, psnrs = [], [], []
        loader = DataLoader(self.eval_dataset, batch_size=self.batch_size, shuffle=False, num_workers=0)
        for batch in loader:
            mri, pet = batch[0].to(self.device), batch[1].to(self.device)
            fused = self.global_model(mri, pet)
            losses.append(loss_fn(fused, mri, pet).item())
            ref = 0.5 * (mri + pet)
            ssims.append(ssim_score_fn(fused, ref))
            psnrs.append(psnr_score_fn(fused, ref))
        return {
            "eval_loss": float(np.mean(losses)),
            "eval_ssim": float(np.mean(ssims)),
            "eval_psnr": float(np.mean(psnrs)),
        }

    # ------------------------------------------------------------------
    # Full federated run
    # ------------------------------------------------------------------
    def run(
        self,
        T: int,
        E: int,
        ssim_score_fn,
        psnr_score_fn,
        log_every: int = 5,
        checkpoint_every: int = 10,
        checkpoint_path: Optional[str] = None,
        tag: str = "run",
    ) -> list:
        K = len(self.client_datasets)
        history = []
        for t in range(1, T + 1):
            t0 = time.time()
            payloads, total_bytes = [], 0
            for k in range(K):
                delta = self._train_client(k, E)
                payload, bytes_sent = self._compress_delta(delta)
                payloads.append(payload)
                total_bytes += bytes_sent
            avg_delta = self._aggregate(payloads)
            for k in self.global_state:
                self.global_state[k] = self.global_state[k] + avg_delta[k]
            self.global_model.load_state_dict(self.global_state)

            row = {
                "tag": tag,
                "round": t,
                "mode": self.mode,
                "round_secs": time.time() - t0,
                "bytes_per_round": int(total_bytes),
                "samples_per_local_epoch": self.samples_per_local_epoch,
                "fipca_rank": self.fipca_rank if self.mode == "fipca" else None,
                "dp_sigma": self.dp_sigma if self.mode == "dpsgd" else None,
            }
            if t % log_every == 0 or t == 1 or t == T:
                row.update(self.evaluate(ssim_score_fn, psnr_score_fn))
                print(
                    f"  [{tag}] r{t:02d}/{T} | "
                    f"loss={row['eval_loss']:.4f} ssim={row['eval_ssim']:.4f} "
                    f"psnr={row['eval_psnr']:.2f}dB | "
                    f"bytes={row['bytes_per_round']/1e6:.1f}MB/rd | "
                    f"{row['round_secs']:.1f}s"
                )
            if checkpoint_path and (t % checkpoint_every == 0 or t == T):
                torch.save(
                    {"state_dict": self.global_state, "round": t, "tag": tag},
                    f"{checkpoint_path}_r{t}.pt",
                )
            history.append(row)
        return history


In [ ]:
# Pin the Round-1 trainer for MedMNIST + model factories (robust to cell re-run order)
FedAvgTrainerMed = FedAvgTrainer
def build_fedsafe(): return FedSafeFuse()
def build_ifcnn():    return IFCNN()
def loss_factory_beta1(): return FusionLoss(beta=1.0)
print('FedSafe-Fuse params:', f'{count_parameters(FedSafeFuse()):,}')


In [ ]:
# ===== MedMNIST 5-seed sweep (resumable; ~15-20 min/seed) =====
def _read_rows(path):
    rows = []
    if os.path.exists(path):
        with open(path) as f:
            rows = list(csv.DictReader(f))
    return rows

def _completed_seeds(rows, n_methods):
    by_seed = {}
    for r in rows:
        by_seed.setdefault(int(r['seed']), set()).add(r['method'])
    return {s for s, ms in by_seed.items() if len(ms) >= n_methods}

def _final_eval(hist):
    le = [r for r in hist if 'eval_loss' in r][-1]
    return float(le['eval_ssim']), float(le['eval_psnr']), int(le['bytes_per_round'])

def _save_rows(rows, cols, local_path, drive_path):
    for path in [local_path, drive_path]:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=cols, extrasaction='ignore'); w.writeheader()
            for r in rows:
                w.writerow(r)

MED_COLS = ['seed', 'method', 'mode', 'ssim', 'psnr', 'bytes_per_round']
MED_LOCAL = f'{PROJECT_DRIVE}/results/medmnist_multiseed.csv'
MED_DRIVE = f'{DRIVE_ROOT}/results/medmnist_multiseed.csv'
# methods in the SAME ORDER as 03_fedtrain (RNG coupling => reproduces seed 42)
MED_METHODS = [('IFCNN+FedAvg',   'ifcnn',   'standard', {}),
               ('FedSafe+DP-SGD', 'fedsafe', 'dpsgd',    {'dp_clip': 1.0, 'dp_sigma': 0.5}),
               ('FedSafe+FROP',   'fedsafe', 'fipca',    {'fipca_rank': 32})]

med_rows = _read_rows(MED_LOCAL) or _read_rows(MED_DRIVE)
done = _completed_seeds(med_rows, len(MED_METHODS))
print('MedMNIST seeds already complete:', sorted(done))

for seed in SEEDS:
    if seed in done:
        print(f'  seed {seed}: skip (already complete)'); continue
    print(f'\n===== MedMNIST seed {seed} =====')
    set_all_seeds(seed)   # seed ONCE per block; methods run back-to-back (mirrors original)
    tseed = time.time()
    for name, arch, mode, extra in MED_METHODS:
        bf = build_ifcnn if arch == 'ifcnn' else build_fedsafe
        tr = FedAvgTrainerMed(build_model_fn=bf, loss_fn_factory=loss_factory_beta1,
                              client_datasets=medmnist_client_datasets,
                              eval_dataset=medmnist_eval_dataset, device=DEVICE, mode=mode,
                              samples_per_local_epoch=40, lr=1e-3, batch_size=16, seed=seed, **extra)
        hist = tr.run(T=50, E=5, ssim_score_fn=ssim_score, psnr_score_fn=psnr_score,
                      log_every=25, checkpoint_every=999, tag=f'{arch}_{mode}_s{seed}')
        ss, ps, bpr = _final_eval(hist)
        med_rows = [r for r in med_rows if not (int(r['seed']) == seed and r['method'] == name)]
        med_rows.append({'seed': seed, 'method': name, 'mode': mode,
                         'ssim': ss, 'psnr': ps, 'bytes_per_round': bpr})
        print(f'  {name:16s} SSIM={ss:.4f} PSNR={ps:6.2f} bytes/rd={bpr}')
    _save_rows(med_rows, MED_COLS, MED_LOCAL, MED_DRIVE)
    print(f'  seed {seed} done in {(time.time()-tseed)/60:.1f} min -> saved (local + Drive)')

# --- seed-42 self-check vs published single-seed SSIM ---
PUB_MED = {'IFCNN+FedAvg': 0.972, 'FedSafe+DP-SGD': 0.235, 'FedSafe+FROP': 0.239}
print('\n--- MedMNIST seed-42 self-check (should match published) ---')
for r in med_rows:
    if int(r['seed']) == 42 and r['method'] in PUB_MED:
        delta = float(r['ssim']) - PUB_MED[r['method']]
        flag = 'OK' if abs(delta) < 0.02 else '*** CHECK ***'
        print(f'  {r["method"]:16s} sweep={float(r["ssim"]):.4f} '
              f'published={PUB_MED[r["method"]]:.3f} delta={delta:+.4f} {flag}')


## Part B — BraTS sweep

In [ ]:
# FedAvg simulator (Round-2 trainer, EF + resampling supported)
"""In-process FedAvg simulator supporting standard, DP-SGD, and FIPCA modes.

A single-process replacement for Flower's SimulationEngine, designed for Colab
Free's compute budget. Mirrors the round structure of Flower so the algorithm
can be swapped to Flower without changing the experimental protocol.

Three modes:
    - 'standard': vanilla FedAvg (raw weight deltas).
    - 'dpsgd'   : DP-SGD on each client (per-batch gradient clip + Gaussian noise),
                  then standard FedAvg aggregation of resulting weight deltas.
    - 'fipca'   : Client weight delta is projected to a rank-d subspace via a
                  pseudo-random Gaussian matrix derived from a fixed seed (shared
                  basis across clients, never transmitted). Server averages the
                  rank-d coefficients and reconstructs. With error_feedback=True
                  each client additionally keeps a local residual of the part of
                  its delta the projection discarded and adds it back into the
                  next round's delta before projecting (memory-compensated
                  compression, Stich et al. 2018). The transmitted payload is
                  still exactly rank-d coefficients and the residual never
                  leaves the client.

Communication cost per round is logged in bytes, which is the primary privacy /
efficiency claim of FedSafe-Fuse.
"""

from __future__ import annotations

import time
from typing import Callable, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset


def _flatten_state(state: dict) -> torch.Tensor:
    """Concatenate all tensors in a state_dict into a single 1-D float32 vector."""
    return torch.cat([v.flatten().float() for v in state.values()])


def _unflatten_to_state(flat: torch.Tensor, template: dict) -> dict:
    """Inverse of _flatten_state, using template state_dict for shapes."""
    out: dict = {}
    idx = 0
    for k, v in template.items():
        n = v.numel()
        out[k] = flat[idx : idx + n].reshape(v.shape).to(v.dtype)
        idx += n
    return out


class FedAvgTrainer:
    """Single-process federated trainer.

    Parameters
    ----------
    build_model_fn :
        Zero-arg callable that returns a fresh model on CPU. Cloned per client.
    loss_fn_factory :
        Zero-arg callable returning a loss module (e.g. FusionLoss(beta)).
        Recreated per client to avoid sharing buffers (SSIM Gaussian window).
    client_datasets :
        List of K Dataset objects, one per client. Items must be (mri, pet, ...).
    eval_dataset :
        Held-out evaluation set (e.g. the calibration split).
    device :
        'cuda' or 'cpu'.
    mode :
        'standard', 'dpsgd', or 'fipca'.
    samples_per_local_epoch :
        Cap on examples drawn per client per local epoch (Round 1 compute cut).
    fipca_rank :
        d, the rank of the FIPCA random-projection subspace.
    error_feedback :
        FIPCA only. If True, each client accumulates the projection residual
        locally and adds it to the next round's delta before compressing.
        NOTE: with a fixed basis this is provably inert — the residual lies in
        the null space of P, so P @ residual = 0 and the payload is unchanged.
        Error feedback only transmits accumulated information when the basis
        changes between rounds; combine with resample_basis=True.
    resample_basis :
        FIPCA only. If True, the orthonormal basis P is regenerated at the
        start of every round from a per-round seed (seed + 7919 * t). The seed
        schedule is deterministic and shared by all clients, so the basis is
        never transmitted and bytes/round are unchanged. Successive subspaces
        are independent, so the error-feedback residual from round t overlaps
        round t+1's subspace and is gradually transmitted.
    dp_clip, dp_sigma :
        DP-SGD per-batch gradient L2-clip threshold and noise multiplier.
    lr :
        Adam learning rate.
    batch_size :
        Per-client batch size.
    seed :
        Random seed for reproducibility.
    """

    VALID_MODES = ("standard", "dpsgd", "fipca")

    def __init__(
        self,
        build_model_fn: Callable[[], nn.Module],
        loss_fn_factory: Callable[[], nn.Module],
        client_datasets: list,
        eval_dataset: Dataset,
        device: str = "cuda",
        mode: str = "standard",
        samples_per_local_epoch: int = 100,
        fipca_rank: int = 32,
        error_feedback: bool = False,
        resample_basis: bool = False,
        dp_clip: float = 1.0,
        dp_sigma: float = 0.5,
        lr: float = 1e-3,
        batch_size: int = 16,
        seed: int = 42,
    ):
        assert mode in self.VALID_MODES, f"mode must be one of {self.VALID_MODES}"
        self.build_model_fn = build_model_fn
        self.loss_fn_factory = loss_fn_factory
        self.client_datasets = client_datasets
        self.eval_dataset = eval_dataset
        self.device = device
        self.mode = mode
        self.samples_per_local_epoch = samples_per_local_epoch
        self.fipca_rank = fipca_rank
        self.error_feedback = error_feedback and mode == "fipca"
        self.resample_basis = resample_basis and mode == "fipca"
        self.dp_clip = dp_clip
        self.dp_sigma = dp_sigma
        self.lr = lr
        self.batch_size = batch_size
        self.seed = seed
        self._rng = np.random.default_rng(seed)

        # Global model (server's authoritative copy)
        self.global_model = build_model_fn().to(device)
        self.global_state = {k: v.detach().clone() for k, v in self.global_model.state_dict().items()}

        # FIPCA: deterministic ORTHONORMAL projection matrix P in (rank, n_params).
        # Built via QR decomposition so that P @ P.T = I_rank exactly, which makes
        # P.T @ P a true rank-d orthogonal projector with operator norm 1.
        # Earlier scaled-Gaussian attempt had operator norm ~sqrt(n_params/rank),
        # causing reconstructed deltas to blow up and Adam to diverge (NaN).
        if mode == "fipca":
            self._n_params = sum(v.numel() for v in self.global_state.values())
            self._build_basis(seed)
            # Error-feedback residuals: one (n_params,) vector per client, kept
            # strictly client-side. Allocated lazily on first compression.
            self._ef_residuals: list = [None] * len(client_datasets)

    def _build_basis(self, basis_seed: int) -> None:
        """(Re)build the rank-d orthonormal projection P from a seed."""
        gen = torch.Generator(device=self.device).manual_seed(basis_seed)
        G = torch.randn(self._n_params, self.fipca_rank, device=self.device, generator=gen)
        Q, _ = torch.linalg.qr(G)  # Q has orthonormal columns: (n_params, rank)
        self.P = Q.T.contiguous()  # (rank, n_params) with orthonormal rows
        del G, Q

    # ------------------------------------------------------------------
    # Client-side local training
    # ------------------------------------------------------------------
    def _local_subset_loader(self, client_idx: int) -> DataLoader:
        ds = self.client_datasets[client_idx]
        n = min(self.samples_per_local_epoch, len(ds))
        sel = self._rng.choice(len(ds), size=n, replace=False)
        subset = Subset(ds, sel.tolist())
        return DataLoader(subset, batch_size=self.batch_size, shuffle=True, num_workers=0)

    def _train_client(self, client_idx: int, n_local_epochs: int) -> dict:
        """Run E local epochs on a fresh random subset of this client's data, return weight delta."""
        local_model = self.build_model_fn().to(self.device)
        local_model.load_state_dict(self.global_state)
        opt = torch.optim.Adam(local_model.parameters(), lr=self.lr, weight_decay=1e-4)
        loss_fn = self.loss_fn_factory().to(self.device)

        local_model.train()
        for _ in range(n_local_epochs):
            loader = self._local_subset_loader(client_idx)
            for batch in loader:
                mri, pet = batch[0].to(self.device), batch[1].to(self.device)
                fused = local_model(mri, pet)
                loss = loss_fn(fused, mri, pet)
                opt.zero_grad()
                loss.backward()
                if self.mode == "dpsgd":
                    torch.nn.utils.clip_grad_norm_(local_model.parameters(), self.dp_clip)
                    for p in local_model.parameters():
                        if p.grad is not None:
                            p.grad.add_(torch.randn_like(p.grad) * self.dp_sigma * self.dp_clip)
                opt.step()

        local_state = local_model.state_dict()
        delta = {k: (local_state[k].detach() - self.global_state[k]).to(self.device) for k in self.global_state}
        return delta

    # ------------------------------------------------------------------
    # Server-side aggregation
    # ------------------------------------------------------------------
    def _compress_delta(self, delta: dict, client_idx: int) -> tuple:
        """Return (payload, bytes_sent) for one client's update."""
        if self.mode == "fipca":
            flat = _flatten_state(delta).to(self.device)
            if self.error_feedback:
                if self._ef_residuals[client_idx] is None:
                    self._ef_residuals[client_idx] = torch.zeros_like(flat)
                flat = flat + self._ef_residuals[client_idx]
            coeffs = self.P @ flat  # (rank,)
            if self.error_feedback:
                # What the server will reconstruct from this client's coeffs;
                # everything the projection dropped is carried to next round.
                recon = self.P.T @ coeffs
                self._ef_residuals[client_idx] = (flat - recon).detach()
            payload = coeffs.detach().clone()
            bytes_sent = payload.numel() * 4  # float32
            return payload, bytes_sent
        else:
            payload = {k: v.detach().clone() for k, v in delta.items()}
            bytes_sent = sum(v.numel() * 4 for v in payload.values())
            return payload, bytes_sent

    def _aggregate(self, client_payloads: list) -> dict:
        if self.mode == "fipca":
            avg_coeffs = torch.stack(client_payloads).mean(dim=0)  # (rank,)
            recon_flat = self.P.T @ avg_coeffs  # (n_params,)
            return _unflatten_to_state(recon_flat, self.global_state)
        else:
            avg = {}
            for k in self.global_state:
                stacked = torch.stack([cp[k] for cp in client_payloads])
                if stacked.dtype.is_floating_point or stacked.dtype.is_complex:
                    avg[k] = stacked.mean(dim=0)
                else:
                    # Integer buffers (e.g. BatchNorm.num_batches_tracked) cannot be
                    # mean()-ed directly; promote to float, mean, and cast back.
                    avg[k] = stacked.to(torch.float32).mean(dim=0).to(stacked.dtype)
            return avg

    # ------------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------------
    @torch.no_grad()
    def evaluate(self, ssim_score_fn, psnr_score_fn) -> dict:
        self.global_model.load_state_dict(self.global_state)
        self.global_model.eval()
        loss_fn = self.loss_fn_factory().to(self.device)
        losses, ssims, psnrs = [], [], []
        loader = DataLoader(self.eval_dataset, batch_size=self.batch_size, shuffle=False, num_workers=0)
        for batch in loader:
            mri, pet = batch[0].to(self.device), batch[1].to(self.device)
            fused = self.global_model(mri, pet)
            losses.append(loss_fn(fused, mri, pet).item())
            ref = 0.5 * (mri + pet)
            ssims.append(ssim_score_fn(fused, ref))
            psnrs.append(psnr_score_fn(fused, ref))
        return {
            "eval_loss": float(np.mean(losses)),
            "eval_ssim": float(np.mean(ssims)),
            "eval_psnr": float(np.mean(psnrs)),
        }

    # ------------------------------------------------------------------
    # Full federated run
    # ------------------------------------------------------------------
    def run(
        self,
        T: int,
        E: int,
        ssim_score_fn,
        psnr_score_fn,
        log_every: int = 5,
        checkpoint_every: int = 10,
        checkpoint_path: Optional[str] = None,
        tag: str = "run",
    ) -> list:
        K = len(self.client_datasets)
        history = []
        for t in range(1, T + 1):
            t0 = time.time()
            if self.mode == "fipca" and self.resample_basis:
                self._build_basis(self.seed + 7919 * t)
            payloads, total_bytes = [], 0
            for k in range(K):
                delta = self._train_client(k, E)
                payload, bytes_sent = self._compress_delta(delta, k)
                payloads.append(payload)
                total_bytes += bytes_sent
            avg_delta = self._aggregate(payloads)
            for k in self.global_state:
                self.global_state[k] = self.global_state[k] + avg_delta[k]
            self.global_model.load_state_dict(self.global_state)

            row = {
                "tag": tag,
                "round": t,
                "mode": self.mode,
                "round_secs": time.time() - t0,
                "bytes_per_round": int(total_bytes),
                "samples_per_local_epoch": self.samples_per_local_epoch,
                "fipca_rank": self.fipca_rank if self.mode == "fipca" else None,
                "error_feedback": self.error_feedback if self.mode == "fipca" else None,
                "resample_basis": self.resample_basis if self.mode == "fipca" else None,
                "dp_sigma": self.dp_sigma if self.mode == "dpsgd" else None,
            }
            if t % log_every == 0 or t == 1 or t == T:
                row.update(self.evaluate(ssim_score_fn, psnr_score_fn))
                print(
                    f"  [{tag}] r{t:02d}/{T} | "
                    f"loss={row['eval_loss']:.4f} ssim={row['eval_ssim']:.4f} "
                    f"psnr={row['eval_psnr']:.2f}dB | "
                    f"bytes={row['bytes_per_round']/1e6:.1f}MB/rd | "
                    f"{row['round_secs']:.1f}s"
                )
            if checkpoint_path and (t % checkpoint_every == 0 or t == T):
                torch.save(
                    {"state_dict": self.global_state, "round": t, "tag": tag},
                    f"{checkpoint_path}_r{t}.pt",
                )
            history.append(row)
        return history


In [ ]:
# Pin the Round-2 trainer for BraTS (redefines FedAvgTrainer above; alias makes driver robust)
FedAvgTrainerBrats = FedAvgTrainer


In [ ]:
# BraTS dataset class (T1=MRI, T2=PET). Rebuilt per seed so max_slices subsampling varies with seed.
from torch.utils.data import Dataset
class BraTSCaseDataset(Dataset):
    def __init__(self, case_ids, root, max_slices=None, rng_seed=42):
        t1s, t2s = [], []; kept = 0
        for cid in case_ids:
            p = f'{root}/brats/processed/{cid}.npz'
            if not os.path.exists(p):
                continue
            with np.load(p) as d:
                t1s.append(d['t1'].astype(np.float32)); t2s.append(d['t2'].astype(np.float32))
            kept += 1
        self.t1 = np.concatenate(t1s, 0); self.t2 = np.concatenate(t2s, 0)
        if max_slices and len(self.t1) > max_slices:
            sel = np.random.default_rng(rng_seed).choice(len(self.t1), max_slices, replace=False)
            self.t1, self.t2 = self.t1[sel], self.t2[sel]
        self.kept_cases = kept
    def __len__(self):
        return len(self.t1)
    def __getitem__(self, i):
        return (torch.from_numpy(self.t1[i]).unsqueeze(0),
                torch.from_numpy(self.t2[i]).unsqueeze(0), 0)


In [ ]:
# ===== BraTS 5-seed sweep (resumable; ~8-15 min/seed) =====
BR_COLS = ['seed', 'method', 'mode', 'ssim', 'psnr', 'bytes_per_round']
BR_LOCAL = f'{PROJECT_DRIVE}/results/brats_multiseed.csv'
BR_DRIVE = f'{DRIVE_ROOT}/results/brats_multiseed.csv'
# methods in the SAME ORDER as 10b_brats (IFCNN -> FROP)
BR_METHODS = [('IFCNN+FedAvg', 'ifcnn',   'standard', {}),
              ('FedSafe+FROP', 'fedsafe', 'fipca',    {'fipca_rank': 32})]

br_rows = _read_rows(BR_LOCAL) or _read_rows(BR_DRIVE)
done_br = _completed_seeds(br_rows, len(BR_METHODS))
print('BraTS seeds already complete:', sorted(done_br))

for seed in SEEDS:
    if seed in done_br:
        print(f'  seed {seed}: skip (already complete)'); continue
    print(f'\n===== BraTS seed {seed} =====')
    set_all_seeds(seed)
    # rebuild datasets with this seed (subsampling varies) -- matches original at seed 42
    br_clients = [BraTSCaseDataset(cl, PROJECT_DRIVE, max_slices=1500, rng_seed=seed)
                  for cl in brats_client_case_lists]
    br_eval = BraTSCaseDataset(brats_calib_case_list, PROJECT_DRIVE, max_slices=512, rng_seed=seed)
    print('  client slices:', [len(ds) for ds in br_clients], '| eval:', len(br_eval))
    tseed = time.time()
    for name, arch, mode, extra in BR_METHODS:
        bf = build_ifcnn if arch == 'ifcnn' else build_fedsafe
        tr = FedAvgTrainerBrats(bf, loss_factory_beta1, br_clients, br_eval, device=DEVICE,
                                mode=mode, samples_per_local_epoch=40, lr=1e-3, batch_size=16,
                                seed=seed, **extra)
        hist = tr.run(T=50, E=5, ssim_score_fn=ssim_score, psnr_score_fn=psnr_score,
                      log_every=25, checkpoint_every=999, tag=f'brats_{arch}_{mode}_s{seed}')
        ss, ps, bpr = _final_eval(hist)
        br_rows = [r for r in br_rows if not (int(r['seed']) == seed and r['method'] == name)]
        br_rows.append({'seed': seed, 'method': name, 'mode': mode,
                        'ssim': ss, 'psnr': ps, 'bytes_per_round': bpr})
        print(f'  {name:16s} SSIM={ss:.4f} PSNR={ps:6.2f} bytes/rd={bpr}')
    _save_rows(br_rows, BR_COLS, BR_LOCAL, BR_DRIVE)
    print(f'  seed {seed} done in {(time.time()-tseed)/60:.1f} min -> saved (local + Drive)')

PUB_BR = {'IFCNN+FedAvg': 0.995, 'FedSafe+FROP': 0.094}
print('\n--- BraTS seed-42 self-check (should match published) ---')
for r in br_rows:
    if int(r['seed']) == 42 and r['method'] in PUB_BR:
        delta = float(r['ssim']) - PUB_BR[r['method']]
        flag = 'OK' if abs(delta) < 0.03 else '*** CHECK ***'
        print(f'  {r["method"]:16s} sweep={float(r["ssim"]):.4f} '
              f'published={PUB_BR[r["method"]]:.3f} delta={delta:+.4f} {flag}')


## Aggregate → mean±std (paste the JSON back to Claude)

In [ ]:
# ===== Aggregate mean+/-std across seeds + emit LaTeX-ready numbers =====
def summarize(path, methods):
    rows = _read_rows(path)
    out = {}
    for name in methods:
        ss = np.array([float(r['ssim']) for r in rows if r['method'] == name])
        ps = np.array([float(r['psnr']) for r in rows if r['method'] == name])
        bpr = sorted({int(r['bytes_per_round']) for r in rows if r['method'] == name})
        out[name] = dict(n=len(ss), ssim_m=ss.mean() if len(ss) else float('nan'),
                         ssim_s=ss.std(ddof=1) if len(ss) > 1 else 0.0,
                         psnr_m=ps.mean() if len(ps) else float('nan'),
                         psnr_s=ps.std(ddof=1) if len(ps) > 1 else 0.0, bytes=bpr)
    return out

med = summarize(f'{PROJECT_DRIVE}/results/medmnist_multiseed.csv',
                ['IFCNN+FedAvg', 'FedSafe+DP-SGD', 'FedSafe+FROP'])
brt = summarize(f'{PROJECT_DRIVE}/results/brats_multiseed.csv',
                ['IFCNN+FedAvg', 'FedSafe+FROP'])

def show(title, summ):
    print(f'\n### {title}')
    for name, s in summ.items():
        print(f'  {name:16s} n={s["n"]}  SSIM={s["ssim_m"]:.3f}+/-{s["ssim_s"]:.3f}  '
              f'PSNR={s["psnr_m"]:.2f}+/-{s["psnr_s"]:.2f}  bytes/rd={s["bytes"]}')
show('MedMNIST (5-seed)', med)
show('BraTS (5-seed)', brt)

print('\n===== PASTE THIS BACK TO CLAUDE =====')
print(json.dumps({'medmnist': med, 'brats': brt}, indent=2, default=float))
